# Решения: Практика: поиск и границы диапазона

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (Path(name), Path("../../data") / name, Path("../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в data/")


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–3. Search и bounds

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


def binary_search(values, target):
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        if values[mid] == target:
            return mid
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


def selection_sort(values):
    result = list(values)
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            if result[j] < result[smallest]:
                smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result


def merge_sorted(left, right):
    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + list(left[i:]) + list(right[j:])


def merge_sort(values):
    if len(values) <= 1:
        return list(values)
    mid = len(values) // 2
    return merge_sorted(merge_sort(values[:mid]), merge_sort(values[mid:]))


def median_runtime(function, values, repeats=3):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(list(values))
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

assert lower_bound([1, 3, 3, 7], 3) == 1
assert upper_bound([1, 3, 3, 7], 3) == 3


## Урок. 4–5. Срезы

In [ ]:
lo, hi = lower_bound(amount_list, 5000), upper_bound(amount_list, 12000)
range_rows = amount_txns[lo:hi]
e_lo = lower_bound(amount_list, max(amount_list) + 1); e_hi = upper_bound(amount_list, max(amount_list) + 100)
empty_rows = amount_txns[e_lo:e_hi]
assert range_rows and empty_rows == []


## Урок. 6–7. Batch и oracle

In [ ]:
targets = [id_list[i] for i in (7, 77, 177, 777)] + [-10]
positions = [binary_search(id_list, target) for target in targets]
probes = [0, 5000, 12000, 50000, 10**9]
bound_checks = [lower_bound(amount_list, x) == sum(v < x for v in amount_list) for x in probes]
assert positions == [7, 77, 177, 777, -1] and all(bound_checks)


## Урок. 8–9. Сложность и gate

In [ ]:
RANGE_NOTE = "Две границы находятся бинарным поиском за O(log n) каждая. Создание ответа требует O(k), где k — число возвращённых строк, поэтому полная стоимость запроса O(log n + k)."
checks = {"lower": lower_bound([], 1) == 0, "upper": upper_bound([1], 1) == 1, "slice": range_rows == [r for r in amount_txns if 5000 <= r[1] <= 12000], "batch": positions[-1] == -1}
assert set(checks.values()) == {True}


## ДЗ. Part A

In [ ]:
count_le_10000 = upper_bound(amount_list, 10000)
queries = [(0, 5000), (10000, 15000), (25000, 35000)]
counts = [upper_bound(amount_list, b) - lower_bound(amount_list, a) for a, b in queries]
a, b = lower_bound(amount_list, 15000), upper_bound(amount_list, 17000)
ids_15_17 = [r[0] for r in amount_txns[a:b]]
assert counts == [sum(a <= x <= b for x in amount_list) for a, b in queries]


## ДЗ. Challenge

In [ ]:
def rows_in_amount_range(rows, amounts, low, high):
    if low > high: return []
    return rows[lower_bound(amounts, low):upper_bound(amounts, high)]

result = rows_in_amount_range(amount_txns, amount_list, 15000, 17000)
bad_result = rows_in_amount_range(amount_txns, amount_list, 20, 10)
RANGE_POLICY = "При low > high функция возвращает пустой список: допустимых значений нет. Эта политика сохраняет тип результата и удобна для pipeline без отдельного исключения."
assert bad_result == [] and len(RANGE_POLICY) >= 140
